# FPINT GEMM Integration Tests

This notebook provides integration tests for FPINT GEMM implementations.
Tests compare all 5 implementations against reference and visualize results.

In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Add parent directory to path for imports
sys.path.insert(0, str(Path('.').absolute()))

import numpy as np
import matplotlib.pyplot as plt

from fpint_emul import *
from visualize import *
from test_utils import *

## Error Functions

Available error functions:
- `relative_diff_percent`: Relative error in percentage (sensitive to small ref values)
- `absolute_diff`: Absolute difference
- `ulp_diff_fp16`: ULP (Units in Last Place) difference for FP16 (recommended for HW verification)

In [ ]:
# ERROR_FUNCS is now imported from test_utils
# Available: 'rel_percent', 'absolute', 'ulp_fp16'
print("Available error functions:", list(ERROR_FUNCS.keys()))

## Helper Functions & Data Classes

In [ ]:
# TestResult now stores 3 types of errors:
# 1. err_ref_vs_gpu:  fpint_gemm_ref (FP64) vs fpint_gemm_gpu (FP16 bit-exact)
# 2. err_ref_vs_emul: fpint_gemm_ref (FP64) vs fpint_emul implementation
# 3. err_ulp_diff:    err_ref_vs_gpu - err_ref_vs_emul
#                     Positive = emul is MORE accurate than gpu
#                     Negative = emul is LESS accurate than gpu

print("TestResult error fields:")
print("  - err_ref_vs_gpu:  ULP error between FP64 ref and GPU (FP16 bit-exact)")
print("  - err_ref_vs_emul: ULP error between FP64 ref and emulation")
print("  - err_ulp_diff:    gpu_err - emul_err (positive = emul better)")
print("\nComputed properties:")
print("  - gpu_max_error, gpu_mean_error")
print("  - emul_max_error, emul_mean_error")
print("  - emul_better_count, emul_worse_count, emul_same_count")

---
# PyVSC Coverage-Driven Testing

Run tests using `pyvsc` library for SystemVerilog-style functional coverage.

**Key concepts:**
- **Constrained Random**: `FpintTestVector` defines randomizable test parameters with constraints
- **Coverage Groups**: Track which scenarios have been tested
- **Cross Coverage**: Track combinations of interesting cases

**Coverage tracks:**
- **Input coverage**: distribution type (normal, mixed_exp, mixed_subnormal), large exp diff, has subnormal
- **Weight coverage**: distribution type (uniform, sparse_zero, has_max), has zero, has max
- **Cross coverage**: max_meets_zero (prealign worst case), sign combinations (++, +-, -+, --)
- **Size coverage**: M, K, N dimensions

**Available functions:**
- `run_fpint_coverage_test()`: Run coverage-driven tests and return runner
- `FpintCoverageTestRunner`: Class for fine-grained control

In [ ]:
# Run coverage-driven tests with store_data=True for debugging
# n_tests: number of random test cases per implementation
runner, analyzer = run_fpint_coverage_test(
    n_tests=20,  # Increase for more thorough testing
    error_threshold=500.0,
    verbose=True,
    store_preds=[
      lambda e: e[ErrorType.REF_VS_GPU] > 10000,
      lambda e: e[ErrorType.REF_VS_EMUL] > 10000
    ]
)

In [ ]:
# Visualize test results and coverage
runner.plot_results()
plt.show()

In [ ]:
analyzer.plot_error_heatmap()

In [ ]:
analyzer.plot_error_histogram()

In [ ]:
analyzer.plot_error_percentile(
  impl_types=[
      FpIntImplType.QCOL_2SCOMP,
      FpIntImplType.QCOL_ZERO_LESS,
      FpIntImplType.QROW_2SCOMP,
      FpIntImplType.QROW_ZERO_LESS,
      FpIntImplType.QROW_REAL_2SCOMP,
  ],
  percentiles=[i*10 for i in range(1, 10)] + [95, 99, 99.9, 99.99, 99.999],
  figsize=(40, 20)
)

In [ ]:
# Use TestResultAnalyzer for detailed analysis
analyzer.print_summary()

# Coverage summary
coverage = runner.get_coverage()
print("\nCoverage Summary:")
for name, value in coverage.items():
    status = "✓" if value >= 80 else "△" if value >= 50 else "✗"
    print(f"  {status} {name}: {value:.1f}%")

# Show worst errors using analyzer
print("\nTop 5 worst errors:")
for r in analyzer.worst_cases(5):
    print(f"  {r}")

---
# Debugging Worst Case Errors

Analyze the test case with maximum error to understand the root cause.

In [ ]:
# Get the worst case (by emul error)
worst = analyzer.worst_cases(1)[0]
print(f"Worst case: {worst}")

# Print detailed comparison
worst.print_comparison()

In [ ]:
# Find the position of max emul error
max_err_pos = np.unravel_index(np.argmax(worst.err_ref_vs_emul), worst.err_ref_vs_emul.shape)
m_idx, n_idx = max_err_pos

print(f"Max emul error position: (m={m_idx}, n={n_idx})")
print(f"\nError values at this position:")
print(f"  GPU error (ref vs gpu):   {worst.err_ref_vs_gpu[m_idx, n_idx]:.2f} ULP")
print(f"  Emul error (ref vs emul): {worst.err_ref_vs_emul[m_idx, n_idx]:.2f} ULP")
print(f"  ULP diff (gpu - emul):    {worst.err_ulp_diff[m_idx, n_idx]:.2f}")

# Show output values
if worst.output_ref_float is not None:
    print(f"\nOutput values:")
    print(f"  Reference (FP64): {worst.output_ref_float[m_idx, n_idx]:.6f}")
    print(f"  GPU (FP16):       {worst.output_gpu_float[m_idx, n_idx]:.6f}")
    print(f"  Emul:             {worst.output_emul_float[m_idx, n_idx]:.6f}")

In [ ]:
# Analyze input row and weight column that caused the max error
if worst.input_data is not None and worst.weight_data is not None:
    input_row = worst.input_data[m_idx, :]  # FP16 bits
    weight_col = worst.weight_data[:, n_idx]  # uint8
    
    # Convert input to float for analysis
    input_float = np.array([fp16_bit_to_float(int(x)) for x in input_row])
    
    print(f"Input row (m={m_idx}):")
    print(f"  Shape: {input_row.shape}")
    print(f"  Float values: min={np.min(input_float):.4f}, max={np.max(input_float):.4f}")
    print(f"  Abs max: {np.max(np.abs(input_float)):.4f} at k={np.argmax(np.abs(input_float))}")
    
    print(f"\nWeight column (n={n_idx}):")
    print(f"  Shape: {weight_col.shape}")
    print(f"  Values: min={np.min(weight_col)}, max={np.max(weight_col)}")
    print(f"  Zero count: {np.sum(weight_col == 0)}")
    print(f"  Max (15) count: {np.sum(weight_col == 15)}")

In [ ]:
# Analyze MXU groups for prealign issues
if worst.input_data is not None and worst.weight_data is not None:
    print("MXU Group Analysis (checking for prealign worst cases):")
    print("=" * 60)
    
    mxu_analysis = analyze_mxu_group(input_row, weight_col, mxu_k=MXU_K)
    
    for group in mxu_analysis:
        g_idx = group['group_idx']
        start_k = g_idx * MXU_K
        end_k = min(start_k + MXU_K, worst.K)
        
        print(f"\nGroup {g_idx} (k={start_k}~{end_k-1}):")
        print(f"  Max exp diff: {group['max_exp_diff']}")
        print(f"  Has subnormal: {group['has_subnormal']}")
        print(f"  Max input idx (in group): {group['max_input_idx']}")
        print(f"  Zero weight indices: {group['zero_weight_indices']}")
        print(f"  MAX_MEETS_ZERO (prealign worst): {group['max_meets_zero']}")
        print(f"  Sign combinations: {group['sign_combinations']}")
        
        if group['max_meets_zero']:
            print(f"  ⚠️  WARNING: Max input multiplied by zero weight!")

In [ ]:
# Visualize all 3 error types for worst case
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Row 1: Three error heatmaps
# 1. GPU error
ax = axes[0, 0]
im = ax.imshow(worst.err_ref_vs_gpu, aspect='auto', cmap='hot')
ax.scatter([n_idx], [m_idx], c='cyan', marker='x', s=100, linewidths=2)
ax.set_title(f'GPU Error (ref vs gpu)\nmax={worst.gpu_max_error:.1f}')
ax.set_xlabel('N')
ax.set_ylabel('M')
plt.colorbar(im, ax=ax, label='ULP')

# 2. Emul error
ax = axes[0, 1]
im = ax.imshow(worst.err_ref_vs_emul, aspect='auto', cmap='hot')
ax.scatter([n_idx], [m_idx], c='cyan', marker='x', s=100, linewidths=2)
ax.set_title(f'Emul Error (ref vs emul)\nmax={worst.emul_max_error:.1f}')
ax.set_xlabel('N')
ax.set_ylabel('M')
plt.colorbar(im, ax=ax, label='ULP')

# 3. ULP diff (positive=emul better, negative=emul worse)
ax = axes[0, 2]
# Use diverging colormap: blue=emul worse, red=emul better
vmax = max(abs(worst.ulp_diff_min), abs(worst.ulp_diff_max))
im = ax.imshow(worst.err_ulp_diff, aspect='auto', cmap='RdBu', vmin=-vmax, vmax=vmax)
ax.scatter([n_idx], [m_idx], c='green', marker='x', s=100, linewidths=2)
ax.set_title(f'ULP Diff (gpu_err - emul_err)\nred=emul better, blue=emul worse')
ax.set_xlabel('N')
ax.set_ylabel('M')
plt.colorbar(im, ax=ax, label='ULP diff')

# Row 2: Output values
if worst.output_ref_float is not None:
    # Reference output
    ax = axes[1, 0]
    im = ax.imshow(worst.output_ref_float, aspect='auto', cmap='viridis')
    ax.scatter([n_idx], [m_idx], c='red', marker='x', s=100, linewidths=2)
    ax.set_title(f'Reference (FP64)\nval={worst.output_ref_float[m_idx, n_idx]:.4f}')
    plt.colorbar(im, ax=ax)

    # GPU output
    ax = axes[1, 1]
    im = ax.imshow(worst.output_gpu_float, aspect='auto', cmap='viridis')
    ax.scatter([n_idx], [m_idx], c='red', marker='x', s=100, linewidths=2)
    ax.set_title(f'GPU (FP16)\nval={worst.output_gpu_float[m_idx, n_idx]:.4f}')
    plt.colorbar(im, ax=ax)

    # Emul output
    ax = axes[1, 2]
    im = ax.imshow(worst.output_emul_float, aspect='auto', cmap='viridis')
    ax.scatter([n_idx], [m_idx], c='red', marker='x', s=100, linewidths=2)
    ax.set_title(f'Emul\nval={worst.output_emul_float[m_idx, n_idx]:.4f}')
    plt.colorbar(im, ax=ax)

plt.suptitle(f'Worst Case Analysis: {worst.name} (emul_better={100*worst.emul_better_count/(worst.M*worst.N):.0f}%)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Detailed input/weight analysis for the max error position
if worst.input_data is not None and worst.weight_data is not None:
    print("Detailed element-wise analysis for max error position:")
    print("=" * 70)
    
    # Show input values with exponents
    print(f"\nInput row (m={m_idx}) - FP16 analysis:")
    print(f"{'k':>4} | {'FP16 bits':>10} | {'Float':>12} | {'Exp':>4} | {'Sign':>4}")
    print("-" * 50)
    for k in range(min(worst.K, 32)):  # Show first 32 elements
        bits = int(input_row[k])
        fval = fp16_bit_to_float(bits)
        exp = get_fp16_exp(bits)
        sign = get_fp16_sign(bits)
        marker = " <-- max" if k == np.argmax(np.abs(input_float)) else ""
        print(f"{k:>4} | 0x{bits:04x}     | {fval:>12.6f} | {exp:>4} | {sign:>4}{marker}")
    
    if worst.K > 32:
        print(f"... ({worst.K - 32} more elements)")

In [ ]:
# Weight column analysis
if worst.weight_data is not None:
    print(f"\nWeight column (n={n_idx}):")
    print(f"{'k':>4} | {'Weight':>6} | {'Input':>12} | {'Product':>12}")
    print("-" * 45)
    for k in range(min(worst.K, 32)):
        w = int(weight_col[k])
        inp = input_float[k]
        prod = inp * w
        zero_marker = " <-- ZERO" if w == 0 else ""
        max_marker = " <-- MAX" if w == 15 else ""
        print(f"{k:>4} | {w:>6} | {inp:>12.6f} | {prod:>12.6f}{zero_marker}{max_marker}")
    
    if worst.K > 32:
        print(f"... ({worst.K - 32} more elements)")

## Reproduce Worst Case with Debug Mode

Re-run the specific worst case with debug=True to see internal computation details.

In [ ]:
# Reproduce the worst case with debug output
# This will show detailed internal computation steps

# Get implementation function based on name
impl_funcs = {
    # 'qcol_2scomp': (fpint_gemm_qcol_2scomp, QCOL),
    # 'qcol_zero_less': (fpint_gemm_qcol_zero_less, QCOL),
    # 'qrow_2scomp': (fpint_gemm_qrow_2scomp, QROW),
    # 'qrow_zero_less': (fpint_gemm_qrow_zero_less, QROW),
    'qrow_real_2scomp': (fpint_gemm_qrow_real_2scomp, QROW),
}

if worst.name in impl_funcs:
    impl_func, qdir = impl_funcs[worst.name]
    
    print(f"Re-running {worst.name} with debug=True")
    print(f"M, K, N -> {worst.M}, {worst.K}, {worst.N}")
    print(f"Focus on m={m_idx}, n={n_idx}")
    print("=" * 70)
    
    # Run with debug (this will print a lot of output)
    # Uncomment below to see full debug output
    debug_output = impl_func(
        worst.input_data[m_idx:m_idx+1, :], 
        worst.weight_data[:, n_idx:n_idx+1], 
        worst.scale_data[:, n_idx//QBLOCK:n_idx//QBLOCK+1], 
        worst.zero_data[:, n_idx//QBLOCK:n_idx//QBLOCK+1],
        1, 
        1, 
        worst.K, 
        debug=True
    )
    
    print("(Debug output disabled by default - uncomment code above to enable)")